# 🏠 House Prices - Regresión Lineal Múltiple (sin Ridge ni Lasso)

# 1. Cargar datos

In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Cargar datos
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# 2. Limpieza de nulos

In [32]:
# bucle para buscar las categoricas
for col in train.select_dtypes(include="object").columns:
    # valores de null a none
    train[col] = train[col].fillna("None")
    # busca en test la misma columna
    if col in test.columns:
        test[col] = test[col].fillna("None")
# bucle para numericos
for col in train.select_dtypes(exclude="object").columns:
    # si vacio rellena con la media
    mediana = train[col].median()
    train[col] = train[col].fillna(mediana)
    # verifica en test la misma columna y mediana de esa columna
    if col in test.columns:
        test[col] = test[col].fillna(mediana)

print("Nulos en train:", train.isnull().sum().sum())
print("Nulos en test:", test.isnull().sum().sum())

Nulos en train: 0
Nulos en test: 0


# 3. Pipeline con pasos para los datos

In [33]:

from sklearn.compose import ColumnTransformer # transformaciones a subconjuntos
from sklearn.preprocessing import OneHotEncoder # oonehotencoder variables categoricas a binarias
from sklearn.impute import SimpleImputer # rellenar valores faltantes
from sklearn.pipeline import Pipeline # creear pasos para los datos

# conjuntos de entrada y salida , identifica categgoricas y numericas 
y = train["SalePrice"]
X = train.drop(columns=["SalePrice", "Id"])
X_test_df = test.drop(columns=["Id"])

num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

# pipeles para cada tipo de dato
# si es numerico
num_pp = Pipeline(steps=[
    # rellena los valores faltantes con la mediana
    ("imputer", SimpleImputer(strategy="median"))
])
# pipeline para categorico
cat_pp = Pipeline(steps=[
    # rellena valoores faltantes con los mas frecuentes
    ("imputer", SimpleImputer(strategy="most_frequent")),
    # convierte columnas a valor numerico, si es descnocodico de otro dataframe  lo ignora
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])
# combinar los pipelines
pre = ColumnTransformer(
    transformers=[
        # aplica los passo de numero a las columnas numericas
        ("num", num_pp, num_cols),
        # aplica los passo de categoria a las columnas categoricas
        ("cat", cat_pp, cat_cols),
    ],
    # columnas no especificadas drop
    remainder="drop"
)



# 4. División de datos y entrenamiento

In [34]:
pipe = Pipeline(steps=[
    # le pasa las columnas y las transforma segun lo que sean
    ("preprocesamiento", pre),
    # con datos procesados sepueden dar al modelo para entrenarlo
    ("modelo", LinearRegression())
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=38
)
# se entrena el modelo primero pasan al pipeline xtrain y ytrin y luego si se entrena
pipe.fit(X_train, y_train)



,steps,"[('preprocesamiento', ...), ('modelo', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


# 5. Evaluación del modelo

In [35]:
y_pred = pipe.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("R2", r2)
print("RMSE", rmse)
print("MSE", mse)


R2 0.8066976474270416
RMSE 35271.25246944831
MSE 1244061250.7635634


# 6. Exportar a Kaggle

In [36]:
pred_final = pipe.predict(X_test_df)

submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": pred_final
})
submission.to_csv("submission9.csv", index=False)
print("submission.csv creado ✅")
submission.head()



submission.csv creado ✅


,Id,SalePrice
0,1461,111429.396179
1,1462,142574.034611
2,1463,179082.007097
3,1464,188757.227936
4,1465,213537.415375
